# Taller de Estatuas 3D · Jaque de Piedra

Convierte las imágenes del Taller de Estatuas en modelos 3D (`.glb`) usando una placa de video gratis de Google.

**Antes de empezar:** menú **Entorno de ejecución → Cambiar tipo de entorno → T4 GPU → Guardar**.

Después tocá ▶ en cada paso, en orden. Los pasos 1 a 3 se hacen una vez por sesión; el paso 4 lo podés repetir con más imágenes.

**Nombrá cada imagen como la pieza** (por ejemplo `rey-realista.png`) y el modelo sale con el mismo nombre.

In [ ]:
#@title 1. Verificar la placa de video
!nvidia-smi --query-gpu=name,memory.total --format=csv


In [ ]:
#@title 2. Instalar Hunyuan3D (unos minutos)
%cd /content
![ -d Hunyuan3D-2 ] || git clone --depth 1 https://github.com/Tencent/Hunyuan3D-2.git
!pip install -q diffusers einops omegaconf trimesh pymeshlab pygltflib rembg onnxruntime
import sys
if '/content/Hunyuan3D-2' not in sys.path: sys.path.insert(0, '/content/Hunyuan3D-2')
print('Instalación lista ✅')


In [ ]:
#@title 3. Cargar la IA (la primera vez baja varios GB)
import torch
from hy3dgen.shapegen import Hunyuan3DDiTFlowMatchingPipeline, FloaterRemover, DegenerateFaceRemover, FaceReducer
from hy3dgen.rembg import BackgroundRemover
pipeline = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained('tencent/Hunyuan3D-2')
rembg = BackgroundRemover()
print('IA lista ✅')


In [ ]:
#@title 4. Subir imágenes y generar las estatuas
#@markdown Podés subir varias imágenes juntas. Al terminar se descarga un .zip con los modelos.
calidad = "alta" #@param ["alta", "rápida"]
import os, time, zipfile
from PIL import Image
from google.colab import files

os.makedirs('/content/estatuas', exist_ok=True)
subidas = files.upload()
hechos = []
for nombre in subidas:
    base = os.path.splitext(nombre)[0]
    print(f'⏳ Generando {base}...')
    t = time.time()
    try:
        img = Image.open(nombre).convert('RGB')
        img = rembg(img)
        mesh = pipeline(
            image=img,
            num_inference_steps=30 if calidad == 'alta' else 15,
            octree_resolution=380 if calidad == 'alta' else 256,
            num_chunks=20000,
            generator=torch.manual_seed(1234),
            output_type='trimesh',
        )[0]
        mesh = FloaterRemover()(mesh)
        mesh = DegenerateFaceRemover()(mesh)
        mesh = FaceReducer()(mesh, max_facenum=30000)
        salida = f'/content/estatuas/{base}.glb'
        mesh.export(salida)
        hechos.append(salida)
        print(f'✅ {base}.glb listo ({time.time() - t:.0f} s)')
    except Exception as e:
        print(f'❌ {base}: {e}')

if hechos:
    zip_path = '/content/estatuas.zip'
    with zipfile.ZipFile(zip_path, 'w') as z:
        for h in hechos:
            z.write(h, os.path.basename(h))
    files.download(zip_path)


In [ ]:
#@title 5. (Opcional) Mirar un modelo en 3D antes de mandarlo
nombre = "rey-realista" #@param {type:"string"}
import trimesh
trimesh.load(f'/content/estatuas/{nombre}.glb').show()
